# Metadados, identificadores e proveniência

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_02/03_metadados_identificadores_e_proveniencia.ipynb)

## 1. Documentação como parte da base

Metadados permitem compreender, localizar, relacionar, administrar e
reutilizar registros. Para esta unidade, distinguiremos:

- **descritivos:** título, autoria, data, assunto;
- **administrativos:** direitos, acesso, formato, responsável;
- **estruturais:** relações entre partes, páginas ou versões;
- **proveniência:** origem, agentes, datas e transformações.

As categorias se sobrepõem em padrões reais; servem aqui como guia de
inspeção.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_02'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 2. Dicionário de dados

Um dicionário deve registrar ao menos nome, definição, tipo, valores ou
regras, origem e limitações. Ele documenta o significado esperado; não
garante que os registros estejam corretos.

In [ ]:
import json
import pandas as pd

catalogo = pd.read_csv("dados/catalogo_fontes.csv")
dicionario = pd.read_csv("dados/dicionario_dados.csv")
dicionario

## 3. Identificadores

Um identificador deve ser único no escopo definido, persistente o bastante
para manter relações e independente de atributos que podem mudar. Título
ou número da linha são candidatos frágeis. O teste abaixo verifica
unicidade e ausência, mas não prova persistência institucional.

In [ ]:
auditoria_id = {
    "registros": len(catalogo),
    "ids_ausentes": int(catalogo["id_fonte"].isna().sum()),
    "ids_duplicados": int(catalogo["id_fonte"].duplicated().sum()),
    "ids_unicos": int(catalogo["id_fonte"].nunique()),
}
auditoria_id

## 4. Campos e domínios esperados

Esta auditoria compara o esquema observado com o documentado e verifica
domínios simples. Diagnosticar não é limpar: decisões de padronização serão
trabalhadas na Unidade 3.

In [ ]:
campos_documentados = set(dicionario["campo"])
campos_observados = set(catalogo.columns)
print("Sem documentação:", sorted(campos_observados - campos_documentados))
print("Documentados e ausentes:", sorted(campos_documentados - campos_observados))

dominios_esperados = {
    "localizado": {"sim", "não"},
    "digitalizado": {"sim", "não"},
    "qualidade_metadados": {"mínimo", "parcial", "completo"},
}
for campo, dominio in dominios_esperados.items():
    inesperados = set(catalogo[campo].dropna()) - dominio
    print(campo, "valores inesperados:", inesperados)

## 5. Proveniência

Proveniência responde: quem criou ou custodiou o registro, de onde ele
veio, quando foi obtido, sob quais condições e que transformações sofreu?
Ela forma uma cadeia entre fonte, representação e resultado. Um endereço
eletrônico sozinho não documenta data de acesso, versão, autoria nem
transformação.

In [ ]:
with open("dados/proveniencia_catalogo.json", encoding="utf-8") as arquivo:
    proveniencia = json.load(arquivo)

for chave in ["titulo", "natureza", "criado_em", "responsavel", "origem"]:
    print(f"{chave}: {proveniencia[chave]}")
print("Transformações registradas:", len(proveniencia["transformacoes"]))

## Atividade — documentação do projeto

**Estratégia de identificadores e escopo de unicidade:** Escreva aqui.

**Metadados mínimos e justificativa:** Escreva aqui.

**Campos do dicionário, definições e domínios:** Escreva aqui.

**Origem, custodiante, versão e data de acesso:** Escreva aqui.

**Transformações previstas e responsável por registrá-las:** Escreva aqui.

**Relação entre registro derivado e fonte:** Escreva aqui.

## Leituras e síntese

Gebru et al. (2021) propõem *datasheets* que documentam motivação,
composição, coleta, usos e manutenção. O modelo PROV-O do W3C oferece
conceitos para representar entidades, atividades e agentes. Eles não
substituem a descrição arquivística ou os padrões específicos do campo;
oferecem perguntas para tornar decisões rastreáveis.

Uma base auditável combina documentação legível por pessoas com
verificações computacionais simples. Dados completos: `referencias.md`.